# Feature Selection + XGBoost Tuning — Puripat (wk2–3)

ต่อจาก `03_xgboost_experiment_P.ipynb` ทำ 3 อย่าง:
1. เลือกฟีเจอร์ใหม่ที่ช่วยจริง (ทำบน train เท่านั้น ไม่แตะ test เพื่อไม่ให้ leak)
2. Tune XGBoost ด้วย Optuna (repeated stratified CV บน train)
3. ประเมินบน test ครั้งเดียว, log เข้า MLflow และ register model

⚠️ ผลนี้เป็นของ Puripat คนเดียว — run สุดท้ายต้องเทียบและตัดสินร่วมกับ Saphondanai ก่อนใช้เป็นโมเดลหลัก

In [1]:
import sys
sys.path.append("../src")
import warnings
warnings.filterwarnings("ignore")
import mlflow, mlflow.xgboost, numpy as np, optuna, pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score, train_test_split
from xgboost import XGBClassifier

from clean_pipeline import RAW_FILENAME, clean_data, load_raw_data
from feature_pipeline import add_features

optuna.logging.set_verbosity(optuna.logging.WARNING)
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("attrition-xgboost-P-tuning")

base = clean_data(load_raw_data(f"../data/raw/{RAW_FILENAME}"))
full = add_features(base)
NEW = [c for c in full.columns if c not in base.columns]
X, y = full.drop(columns="Attrition"), full["Attrition"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
spw = (ytr == 0).sum() / (ytr == 1).sum()
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
DEFAULT = dict(n_estimators=300, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
               random_state=42, scale_pos_weight=spw, n_jobs=4)

def cv_auc(cols, params=DEFAULT):
    return cross_val_score(XGBClassifier(**params), Xtr[cols], ytr, cv=cv, scoring="roc_auc").mean()

print("ฟีเจอร์ใหม่:", NEW, "| train:", Xtr.shape, "test:", Xte.shape)

2026/09/30 00:24:19 INFO mlflow.tracking.fluent: Experiment with name 'attrition-xgboost-P-tuning' does not exist. Creating a new experiment.


ฟีเจอร์ใหม่: ['TenureRatio', 'IncomePerLevel', 'PromotionGap', 'ManagerStability', 'JobHopping', 'AvgSatisfaction', 'OverTimeXDistance'] | train: (1176, 54) test: (294, 54)


## 1. เลือกฟีเจอร์: เพิ่มทีละตัวเข้าชุด base แล้วดู CV AUC บน train

In [2]:
BASE = [c for c in X.columns if c not in NEW]
ref = cv_auc(BASE)
gain = pd.Series({f: cv_auc(BASE + [f]) - ref for f in NEW}).sort_values(ascending=False)
print(f"base CV AUC = {ref:.4f}")
gain.round(4)

base CV AUC = 0.7966


OverTimeXDistance    0.0026
AvgSatisfaction      0.0016
TenureRatio          0.0007
IncomePerLevel      -0.0001
JobHopping          -0.0021
PromotionGap        -0.0029
ManagerStability    -0.0031
dtype: float64

เก็บเฉพาะตัวที่ช่วยเพิ่ม CV AUC (gain > 0) แล้ววัดชุดรวมอีกครั้งเทียบกับ base และชุดฟีเจอร์ครบ

In [3]:
KEEP = gain[gain > 0].index.tolist()
sel = {"base": BASE, "base+selected": BASE + KEEP, "base+all_new": BASE + NEW}
sel_auc = {k: cv_auc(v) for k, v in sel.items()}
print("เลือก:", KEEP)
pd.Series(sel_auc).round(4)

เลือก: ['OverTimeXDistance', 'AvgSatisfaction', 'TenureRatio']


base             0.7966
base+selected    0.8018
base+all_new     0.8011
dtype: float64

In [4]:
FEATURES = max(sel, key=lambda k: sel_auc[k])
COLS = sel[FEATURES]
print("ใช้ชุด:", FEATURES, f"({len(COLS)} ฟีเจอร์)")

ใช้ชุด: base+selected (50 ฟีเจอร์)


## 2. Tune ด้วย Optuna (40 trials, maximize CV AUC บน train)

In [5]:
def objective(t):
    p = dict(DEFAULT,
        n_estimators=t.suggest_int("n_estimators", 100, 600, step=50),
        max_depth=t.suggest_int("max_depth", 2, 6),
        learning_rate=t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=t.suggest_int("min_child_weight", 1, 10),
        reg_lambda=t.suggest_float("reg_lambda", 0.1, 10, log=True),
    )
    return cv_auc(COLS, p)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40)
print(f"best CV AUC = {study.best_value:.4f} (default = {sel_auc[FEATURES]:.4f})")
study.best_params

best CV AUC = 0.8272 (default = 0.8018)


{'n_estimators': 500,
 'max_depth': 2,
 'learning_rate': 0.03132429398213804,
 'subsample': 0.664952951683747,
 'colsample_bytree': 0.5305253930212119,
 'min_child_weight': 10,
 'reg_lambda': 1.2338780559789992}

## 3. ประเมินบน test (ครั้งเดียว) + log MLflow + register

In [6]:
BEST = dict(DEFAULT, **study.best_params)
def evaluate(params, name):
    m = XGBClassifier(**params).fit(Xtr[COLS], ytr)
    p = m.predict_proba(Xte[COLS])[:, 1]
    return m, {"test_auc": roc_auc_score(yte, p), "test_pr_auc": average_precision_score(yte, p),
               "test_f1": f1_score(yte, p > 0.5)}

_, m_def = evaluate(DEFAULT, "default")
model, m_best = evaluate(BEST, "tuned")
with mlflow.start_run(run_name="xgb-tuned-P"):
    mlflow.log_params({**study.best_params, "scale_pos_weight": round(spw, 2), "feature_set": FEATURES, "n_features": len(COLS)})
    mlflow.log_metrics({**m_best, "cv_auc": study.best_value})
    info = mlflow.xgboost.log_model(model, name="model", registered_model_name="attrition-xgboost-P")
pd.DataFrame({"default": m_def, "tuned": m_best}).round(3)

Successfully registered model 'attrition-xgboost-P'.
Created version '1' of model 'attrition-xgboost-P'.


,default,tuned
test_auc,0.770,0.812
test_pr_auc,0.503,0.595
test_f1,0.485,0.467


## สรุป / สิ่งที่ต้องทำต่อ
- ชุดฟีเจอร์และ hyperparameter ข้างบนคือ **ข้อเสนอของ Puripat** สำหรับเทียบกับ run ของ Saphondanai
- register ไว้ใน MLflow ในเครื่อง (`mlflow.db`) เท่านั้น — ต้องย้ายไป tracking server กลางเมื่อ Saphondanai แจก URI
- test set ใช้ประเมินครั้งเดียว ห้ามนำไปเลือกฟีเจอร์/พารามิเตอร์เพิ่ม